# Bài tập: Xây dựng mạng ANN phân loại chữ số viết tay MNIST

Bài này thực hiện việc xây dựng một mạng Neural Network (ANN) để nhận diện các chữ số từ 0 đến 9 trong bộ dữ liệu MNIST sử dụng PyTorch.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix
import seaborn as sns

# 1. Cấu hình thiết bị (Ưu tiên GPU nếu chạy trên Colab)
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"🚀 Đang chạy trên GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print("💻 Đang chạy trên GPU Apple Silicon (MPS)")
else:
    device = torch.device('cpu')
    print("⚠️ Đang chạy trên CPU (Sẽ chậm hơn đó fen)")

# Thiết lập seed để kết quả ổn định
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

## 1. Chuẩn bị dữ liệu

Chúng ta sẽ tải bộ dữ liệu MNIST và chuẩn hóa chúng.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Số lượng tập huấn luyện: {len(train_dataset)}")
print(f"Số lượng tập kiểm tra: {len(test_dataset)}")

## 2. Định nghĩa mô hình ANN

Chúng ta sẽ xây dựng một mạng ANN với 2 lớp ẩn (theo bài tập nâng cao).

In [ ]:
class ImprovedANN(nn.Module):
    def __init__(self, input_size=784, hidden_size1=256, hidden_size2=128, num_classes=10):
        super(ImprovedANN, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(input_size, hidden_size1)
        self.relu1 = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.fc2 = nn.Linear(hidden_size1, hidden_size2)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden_size2, num_classes)
        
    def forward(self, x):
        x = self.flatten(x)
        x = self.relu1(self.fc1(x))
        x = self.dropout(x)
        x = self.relu2(self.fc2(x))
        x = self.fc3(x)
        return x

model = ImprovedANN().to(device)
print(model)

## 3. Huấn luyện mô hình

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train_model(model, train_loader, test_loader, epochs=10):
    train_losses, test_losses = [], []
    train_accs, test_accs = [], []
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = correct / total
        
        # Đánh giá trên tập test
        model.eval()
        test_loss = 0.0
        test_correct = 0
        test_total = 0
        
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                test_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs.data, 1)
                test_total += labels.size(0)
                test_correct += (predicted == labels).sum().item()
                
        test_epoch_loss = test_loss / len(test_loader.dataset)
        test_epoch_acc = test_correct / test_total
        
        train_losses.append(epoch_loss)
        test_losses.append(test_epoch_loss)
        train_accs.append(epoch_acc)
        test_accs.append(test_epoch_acc)
        
        print(f'Epoch {epoch+1}/{epochs} | Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f} | Test Loss: {test_epoch_loss:.4f} Acc: {test_epoch_acc:.4f}')
        
    return train_losses, test_losses, train_accs, test_accs

train_losses, test_losses, train_accs, test_accs = train_model(model, train_loader, test_loader)

## 4. Trực quan hóa kết quả

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(test_losses, label='Test Loss')
plt.title('Loss History')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accs, label='Train Acc')
plt.plot(test_accs, label='Test Acc')
plt.title('Accuracy History')
plt.legend()
plt.show()

## 5. Ma trận nhầm lẫn (Confusion Matrix)

In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Dự đoán')
plt.ylabel('Thực tế')
plt.title('Confusion Matrix')
plt.show()